# 05 — State backend smoke test (Cosmos DB or LocalState)

Verifies CRUD on every container and that the partition keys work.

Uses `create_state_service()` so the notebook works against either:
- **CosmosService** — when `COSMOS_ENDPOINT` and `COSMOS_KEY` are set in `.env`.
- **LocalStateService** — file-backed JSON fallback (no Cosmos required).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.cosmos_client import create_state_service
from src.models import ChatTurn, DocumentMeta, FeedbackRecord, LearnedRule, GoldenPair

# Returns CosmosService if COSMOS_ENDPOINT/COSMOS_KEY are set, else LocalStateService
# (file-backed JSON at LOCAL_STATE_PATH, default /app/data/state.json).
cosmos = create_state_service()
cosmos.ensure_containers()
print('Backend:', type(cosmos).__name__)

In [ ]:
# Sessions
t1 = ChatTurn(session_id='s-test', role='user', content='hi')
t2 = ChatTurn(session_id='s-test', role='assistant', content='hello')
cosmos.save_turn(t1); cosmos.save_turn(t2)
for h in cosmos.get_history('s-test'):
    print(h.role, ':', h.content)

In [ ]:
# Documents
doc = DocumentMeta(user_id='nb-user', filename='t.pdf', blob_name='nb/t.pdf')
cosmos.save_document(doc)
print(cosmos.list_documents('nb-user'))

In [ ]:
# Feedback + rules + golden
fb = FeedbackRecord(session_id='s-test', turn_id=t2.id, rating='down', correction='be more concise', question='hi', answer='hello')
cosmos.save_feedback(fb)
cosmos.save_rule(LearnedRule(category='general', rule='Always be concise'))
cosmos.save_golden(GoldenPair(topic='general', question='hi', answer='hello'))
print('rules:', cosmos.get_rules())
print('golden:', cosmos.get_golden_pairs())

In [ ]:
# Chunk quality
cosmos.update_chunk_quality('chunk-x', retrieved=True, good=True)
cosmos.update_chunk_quality('chunk-x', good=True)
cosmos.update_chunk_quality('chunk-x', bad=True)
print(cosmos.get_chunk_quality('chunk-x'))

In [ ]:
# Cleanup test doc
cosmos.delete_document(doc.id, user_id='nb-user')
print('cleaned ✔')